<a href="https://colab.research.google.com/github/AnamikaMangore/anamika-ka-project/blob/txt_classifer/Text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

In [ ]:
train_data, validation_data,test_data = tfds.load(name="imdb_reviews", split=('train[:60%]','train[60%:]','test'), as_supervised=True)

In [ ]:
train_data

<_PrefetchDataset element_spec=(TensorSpec(shape=(), dtype=tf.string, name=None), TensorSpec(shape=(), dtype=tf.int64, name=None))>

In [ ]:
train_examples_batch, train_labels_batch = next(iter(train_data.batch(10)))
train_examples_batch

<tf.Tensor: shape=(10,), dtype=string, numpy=
array([b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it.",
       b'I have been known to fall asleep during films, but this is usually due to a combination of things including, really tired, being warm and comfortable on the sette and having just eaten a lot. However on this occasion I fell 

In [ ]:
train_labels_batch

<tf.Tensor: shape=(10,), dtype=int64, numpy=array([0, 0, 0, 1, 1, 1, 0, 0, 0, 0])>

In [ ]:
embedding="https://tfhub.dev/google/nnlm-en-dim50/2"

In [ ]:
hub_layer=hub.KerasLayer(embedding,input_shape=[],dtype=tf.string,trainable=True)
hub_layer(train_examples_batch)

<tf.Tensor: shape=(10, 50), dtype=float32, numpy=
array([[ 5.42319477e-01, -1.19017037e-02,  6.33753762e-02,
         6.86297193e-02, -1.67768374e-01, -1.05811745e-01,
         1.68653026e-01, -4.99882400e-02, -3.11480552e-01,
         7.91034624e-02,  1.54422626e-01,  1.48866158e-02,
         3.93015295e-02,  1.97727114e-01, -1.22154757e-01,
        -4.12098095e-02, -2.70410895e-01, -2.19221517e-01,
         2.65176624e-01, -8.07390749e-01,  2.58335322e-01,
        -3.10042113e-01,  2.86832154e-01,  1.94338694e-01,
        -2.90364921e-01,  3.86284851e-02, -7.84441113e-01,
        -4.79324013e-02,  4.11029905e-01, -3.63888919e-01,
        -5.80347061e-01,  3.02694559e-01,  3.63089710e-01,
        -1.52271643e-01, -4.43915039e-01,  1.94629967e-01,
         1.95284083e-01,  5.66623397e-02,  2.89070398e-01,
        -2.84683228e-01, -5.31205582e-03,  5.71938045e-02,
        -3.20131809e-01, -4.41866480e-02, -8.55078250e-02,
        -5.58474362e-01, -2.33363912e-01, -2.07829520e-01,
      

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

# ✅ Load the Hub Layer from TensorFlow Hub
hub_layer = hub.KerasLayer("https://tfhub.dev/google/nnlm-en-dim50/2",
                           dtype=tf.string, trainable=True)

# ✅ Wrap the Hub Layer in a Custom Keras Layer
class HubWrapperLayer(tf.keras.layers.Layer):
    def __init__(self, hub_layer):
        super(HubWrapperLayer, self).__init__()
        self.hub_layer = hub_layer

    def call(self, inputs):
        return self.hub_layer(inputs)

# ✅ Now use Functional API without error
inputs = tf.keras.layers.Input(shape=(), dtype=tf.string)
x = HubWrapperLayer(hub_layer)(inputs)  # ✅ This will work perfectly now
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(32, activation='relu')(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

# ✅ Build the Model
model = tf.keras.Model(inputs=inputs, outputs=outputs)

# ✅ Compile the Model
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Model Summary
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)           │ (None)                      │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ hub_wrapper_layer_1                  │ (None, 50)                  │               0 │
│ (HubWrapperLayer)                    │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           3,264 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 5,377 (21.00 KB)

 Trainable params: 5,377 (21.00 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              metrics=['accuracy'])


In [35]:
history=model.fit(train_data.shuffle(10000).batch(1024),
                    epochs=100,
                    validation_data=validation_data.batch(1024),
                    verbose=1)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 101ms/step - accuracy: 0.7759 - loss: 0.4737 - val_accuracy: 0.7619 - val_loss: 0.5012
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - accuracy: 0.7778 - loss: 0.4715 - val_accuracy: 0.7636 - val_loss: 0.5010
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 0.7754 - loss: 0.4730 - val_accuracy: 0.7636 - val_loss: 0.5009
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 98ms/step - accuracy: 0.7722 - loss: 0.4713 - val_accuracy: 0.7603 - val_loss: 0.5012
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 146ms/step - accuracy: 0.7746 - loss: 0.4704 - val_accuracy: 0.7616 - val_loss: 0.5011
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 3s 144ms/step - accuracy: 0.7767 - loss: 0.4749 - val_accuracy: 0.7634 - val_loss: 0.5013
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 146ms/step - accuracy: 0.7768 - loss: 0.4664 - val_accuracy: 0.7622 - val_loss: 0.5015
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 2s 106ms/step - accuracy: 0.7726 - loss: 0.4781 - val_accura

In [36]:
results= model.evaluate(test_data.batch(512),verbose=2)
for name,value in zip(model.metrics_names,results):
    print(name,":",value)
print("Done")

49/49 - 4s - 84ms/step - accuracy: 0.7441 - loss: 0.5216
loss : 0.521636426448822
compile_metrics : 0.7440800070762634
Done


In [37]:
# prompt: how to increase accuracy i want accuracy above 90%

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_datasets as tfds

# Load the IMDB Reviews dataset
train_data, validation_data, test_data = tfds.load(
    name="imdb_reviews",
    split=('train[:60%]', 'train[60%:]', 'test'),
    as_supervised=True
)

# Preprocessing: Text vectorization layer
vectorize_layer = tf.keras.layers.TextVectorization(
    max_tokens=10000,  # Adjust based on your vocabulary size
    output_mode='int',
    output_sequence_length=512  # Adjust based on sequence length
)

# Adapt the vectorization layer to the training data
vectorize_layer.adapt(train_data.map(lambda text, label: text))

# Embedding layer
embedding_dim = 128  # Increased embedding dimension
embedding_layer = tf.keras.layers.Embedding(10000, embedding_dim)

# Model definition
model = tf.keras.Sequential([
    vectorize_layer,
    embedding_layer,
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(128, activation='relu'),  # Increased neurons
    tf.keras.layers.Dropout(0.5),  # Added dropout for regularization
    tf.keras.layers.Dense(1, activation='sigmoid')
])


# Compile the model
model.compile(optimizer='adam',
              loss=tf.keras.losses.BinaryCrossentropy(from_logits=False), # removed from_logits=True
              metrics=['accuracy'])

# Train the model
history = model.fit(
    train_data.batch(512),  # Increased batch size
    epochs=20,  # Increased epochs
    validation_data=validation_data.batch(512),
    verbose=1
)

# Evaluate the model
results = model.evaluate(test_data.batch(512), verbose=2)
for name, value in zip(model.metrics_names, results):
    print(f"{name}: {value}")
print("Done")


Epoch 1/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 16s 486ms/step - accuracy: 0.5003 - loss: 0.6937 - val_accuracy: 0.5328 - val_loss: 0.6892
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 20s 468ms/step - accuracy: 0.5549 - loss: 0.6857 - val_accuracy: 0.5819 - val_loss: 0.6680
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 17s 568ms/step - accuracy: 0.6057 - loss: 0.6597 - val_accuracy: 0.5908 - val_loss: 0.6413
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 18s 471ms/step - accuracy: 0.6402 - loss: 0.6206 - val_accuracy: 0.6093 - val_loss: 0.6192
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 467ms/step - accuracy: 0.6529 - loss: 0.6023 - val_accuracy: 0.7287 - val_loss: 0.5444
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 14s 466ms/step - accuracy: 0.7514 - loss: 0.5230 - val_accuracy: 0.8192 - val_loss: 0.4634
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 21s 465ms/step - accuracy: 0.7930 - loss: 0.4620 - val_accuracy: 0.8016 - val_loss: 0.4443
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 16s 518ms/step - accuracy: 0.7854 - loss: 0.4554 - val_accu